## Load data

### Subtask:
Load the training and testing data from the provided CSV files.


In [ ]:
import pandas as pd

train_df = pd.read_csv('/content/train.csv')
test_df = pd.read_csv('/content/test.csv')
test_labels_df = pd.read_csv('/content/test_labels.csv')

display(train_df.head())
display(test_df.head())
display(test_labels_df.head())

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


,id,comment_text
0,00001cee341fdb12,Yo bitch Ja Rule is more succesful then you'll...
1,0000247867823ef7,== From RfC == \n\n The title is fine as it is...
2,00013b17ad220c46,""" \n\n == Sources == \n\n * Zawe Ashton on Lap..."
3,00017563c3f7919a,":If you have a look back at the source, the in..."
4,00017695ad8997eb,I don't anonymously edit articles at all.


,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,00001cee341fdb12,-1,-1,-1,-1,-1,-1
1,0000247867823ef7,-1,-1,-1,-1,-1,-1
2,00013b17ad220c46,-1,-1,-1,-1,-1,-1
3,00017563c3f7919a,-1,-1,-1,-1,-1,-1
4,00017695ad8997eb,-1,-1,-1,-1,-1,-1


## Preprocess data

### Subtask:
Clean and preprocess the text data. This includes handling missing values, tokenization, and potentially creating sequences for different models.
**Reasoning**:
Handle missing values in the 'comment_text' column for both dataframes.



In [ ]:
train_df['comment_text'].fillna('', inplace=True)
test_df['comment_text'].fillna('', inplace=True)

display(train_df.isnull().sum())
display(test_df.isnull().sum())

/tmp/ipython-input-1669901378.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df['comment_text'].fillna('', inplace=True)
/tmp/ipython-input-1669901378.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try u

,0
id,0
comment_text,0
toxic,0
severe_toxic,0
obscene,0
threat,0
insult,0
identity_hate,0


,0
id,0
comment_text,0


**Reasoning**:
Perform basic text cleaning on the 'comment_text' column for both dataframes.



In [ ]:
import re
import string

def clean_text(text):
    text = text.lower()
    text = re.sub(f'[{re.escape(string.punctuation)}]', '', text)
    text = re.sub(r'\d+', '', text)
    return text

train_df['comment_text'] = train_df['comment_text'].apply(clean_text)
test_df['comment_text'] = test_df['comment_text'].apply(clean_text)

display(train_df.head())
display(test_df.head())

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,explanation\nwhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,daww he matches this background colour im seem...,0,0,0,0,0,0
2,000113f07ec002fd,hey man im really not trying to edit war its j...,0,0,0,0,0,0
3,0001b41b1c6bb37e,\nmore\ni cant make any real suggestions on im...,0,0,0,0,0,0
4,0001d958c54c6e35,you sir are my hero any chance you remember wh...,0,0,0,0,0,0


,id,comment_text
0,00001cee341fdb12,yo bitch ja rule is more succesful then youll ...
1,0000247867823ef7,from rfc \n\n the title is fine as it is imo
2,00013b17ad220c46,\n\n sources \n\n zawe ashton on lapland —...
3,00017563c3f7919a,if you have a look back at the source the info...
4,00017695ad8997eb,i dont anonymously edit articles at all


**Reasoning**:
Tokenize the cleaned text data and convert the tokenized text into sequences of a fixed length.



In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 20000
maxlen = 100

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(train_df['comment_text'])

train_sequences = tokenizer.texts_to_sequences(train_df['comment_text'])
test_sequences = tokenizer.texts_to_sequences(test_df['comment_text'])

train_padded_sequences = pad_sequences(train_sequences, maxlen=maxlen)
test_padded_sequences = pad_sequences(test_sequences, maxlen=maxlen)

print("Training sequences shape:", train_padded_sequences.shape)
print("Testing sequences shape:", test_padded_sequences.shape)

Training sequences shape: (159571, 100)
Testing sequences shape: (153164, 100)


## Handle class imbalance

### Subtask:
Implement techniques like SMOTE or ADASYN to address the class imbalance in the toxic datasets.
**Reasoning**:
Extract the target labels from the training data and prepare them for oversampling.



In [ ]:
import numpy as np

target_columns = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
train_labels = train_df[target_columns].values

print("Training labels shape:", train_labels.shape)

Training labels shape: (159571, 6)


**Reasoning**:
Apply SMOTE to the padded training sequences and the corresponding target labels to address class imbalance.



**Reasoning**:
SMOTE does not support multilabel targets. To address this, I will apply SMOTE to each target column individually.



In [ ]:
from imblearn.over_sampling import SMOTE
import pandas as pd

target_columns = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
train_padded_sequences_resampled = train_padded_sequences.copy()
train_labels_resampled = train_labels.copy()

for i, col in enumerate(target_columns):
    print(f"Applying SMOTE for column: {col}")
    smote = SMOTE(random_state=42)
    # Resample only the current target column and the corresponding features
    X_resampled_col, y_resampled_col = smote.fit_resample(train_padded_sequences_resampled, train_labels_resampled[:, i])

    # Update the resampled dataframes. This approach can be complex due to
    # potential introduction of new samples that need to be consistent across labels.
    # A simpler approach for multilabel is to treat each label as a separate binary problem
    # and resample independently, then combine. However, this can create inconsistent samples.

    # A more common approach for multilabel is to use techniques designed for it,
    # or to treat it as multiple binary problems and train separate models or
    # use a model that can handle multilabel directly.

    # Given the constraint to use SMOTE/ADASYN, and the error,
    # a practical workaround is to oversample based on the most imbalanced label or
    # to use a strategy that samples minority classes across labels, which SMOTE doesn't do directly for multilabel.

    # Let's try resampling based on the 'toxic' label as it's likely the most frequent minority class.
    # This is a simplification and might not optimally balance all labels.

# Re-attempting with a focus on the primary 'toxic' label for demonstration purposes with SMOTE limitation
print("Applying SMOTE based on the 'toxic' label")
smote = SMOTE(random_state=42)
train_padded_sequences_resampled, train_labels_resampled_toxic = smote.fit_resample(
    train_padded_sequences, train_labels[:, 0] # Resample based on the first target column ('toxic')
)

# For other labels, the resampling based on 'toxic' will create new samples.
# We need to decide how to assign labels for these new samples for the other toxicity types.
# A simple approach is to keep the original labels for the original samples and
# assign a placeholder or use a more advanced multilabel oversampling technique not directly SMOTE.
# Given the tools, we'll proceed with the oversampling based on 'toxic' and acknowledge this limitation.

print("Resampled training sequences shape (based on 'toxic'):", train_padded_sequences_resampled.shape)
print("Resampled training labels shape (toxic only):", train_labels_resampled_toxic.shape)

# To use this for multilabel training, we would need a strategy for the other labels.
# A common but imperfect approach is to assume the new samples generated based on
# 'toxic' oversampling should have their original labels for other categories.
# This requires careful indexing and merging, which is complex with SMOTE's output.

# Let's verify the class distribution for the 'toxic' label after resampling.
unique, counts = np.unique(train_labels_resampled_toxic, return_counts=True)
print("Class distribution for 'toxic' label after SMOTE:", dict(zip(unique, counts)))

# Note: This only addresses the 'toxic' label imbalance directly using SMOTE.
# Addressing the imbalance for all labels simultaneously with standard SMOTE is not straightforward.
# More advanced techniques or different strategies would be needed for a comprehensive multilabel oversampling.


Applying SMOTE for column: toxic
Applying SMOTE for column: severe_toxic
Applying SMOTE for column: obscene
Applying SMOTE for column: threat
Applying SMOTE for column: insult
Applying SMOTE for column: identity_hate
Applying SMOTE based on the 'toxic' label
Resampled training sequences shape (based on 'toxic'): (288554, 100)
Resampled training labels shape (toxic only): (288554,)
Class distribution for 'toxic' label after SMOTE: {np.int64(0): np.int64(144277), np.int64(1): np.int64(144277)}


**Reasoning**:
Verify the class distribution of the other target labels after applying SMOTE based on the 'toxic' label to understand the impact on the overall multilabel distribution.



In [ ]:
# Since SMOTE was applied only based on the 'toxic' label,
# the other labels for the newly generated samples are not automatically determined.
# To properly handle multilabel oversampling, dedicated techniques are needed.
# However, given the constraints and the use of SMOTE, we will examine the distribution
# of the original labels and acknowledge that the oversampling only directly balanced 'toxic'.

print("\nOriginal class distribution for each target label:")
for col in target_columns:
    print(f"{col}:")
    display(train_df[col].value_counts())

# Note: The resampled data (train_padded_sequences_resampled and train_labels_resampled_toxic)
# only represents the oversampling for the 'toxic' label.
# To use this for training a multilabel model, a strategy would be needed to assign
# labels for the other toxicity types to the newly generated samples.
# This is a limitation of using standard SMOTE for multilabel data.



Original class distribution for each target label:
toxic:


,count
toxic,
0,144277
1,15294


severe_toxic:


,count
severe_toxic,
0,157976
1,1595


obscene:


...

,count
threat,
0,159093
1,478


insult:


,count
insult,
0,151694
1,7877


identity_hate:


,count
identity_hate,
0,158166
1,1405


## Implement gru model

### Subtask:
Build and train a GRU model for toxicity classification.


**Reasoning**:
Import necessary Keras layers and define, compile, and train the GRU model.



In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout

# Define the GRU model
gru_model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=maxlen),
    GRU(units=64),
    Dropout(0.5),
    Dense(units=6, activation='sigmoid')
])

# Compile the model
gru_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Assuming train_padded_sequences and train_labels are available from previous steps
# And acknowledging the limitations of SMOTE for multilabel data from the previous step,
# we will proceed with the original train_padded_sequences and train_labels for training
# the multilabel GRU model.

# Train the model
history_gru = gru_model.fit(
    train_padded_sequences,
    train_labels,
    epochs=5,  # Training for 5 epochs as an example
    batch_size=32,
    validation_split=0.2
)

gru_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/5
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 471s 117ms/step - accuracy: 0.7558 - loss: 0.1000 - val_accuracy: 0.9941 - val_loss: 0.0494
Epoch 2/5
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 540s 126ms/step - accuracy: 0.8669 - loss: 0.0460 - val_accuracy: 0.9887 - val_loss: 0.0474
Epoch 3/5
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 459s 115ms/step - accuracy: 0.8323 - loss: 0.0384 - val_accuracy: 0.9884 - val_loss: 0.0499
Epoch 4/5
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 452s 113ms/step - accuracy: 0.7605 - loss: 0.0326 - val_accuracy: 0.9530 - val_loss: 0.0536
Epoch 5/5
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 435s 109ms/step - accuracy: 0.6341 - loss: 0.0276 - val_accuracy: 0.9758 - val_loss: 0.0587


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,792,916 (29.73 MB)

 Trainable params: 2,597,638 (9.91 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 5,195,278 (19.82 MB)

**Reasoning**:
The subtask is to load the data from the CSV file into a pandas DataFrame and display the first few rows.



In [ ]:
import os

# List files in the current directory
print(os.listdir('.'))

['.config', 'train.csv', 'test.csv', 'test_labels.csv', 'sample_data']


**Reasoning**:
The previous command listed the files in the current directory, and 'train.csv' appears to be a relevant data file. I will load this file into a pandas DataFrame and display the first few rows.



In [ ]:
# Load the dataframe from the correct path.
df = pd.read_csv('train.csv')

# Display the first 5 rows.
display(df.head().to_markdown(index=False, numalign="left", stralign="left"))

'| id               | comment_text                                                                                                                                                                                                                                                                                                                                                                                                                       | toxic   | severe_toxic   | obscene   | threat   | insult   | identity_hate   |\n|:-----------------|:-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:--------|:---------------|:--------

## Implement CNN + LSTM Hybrid Model

### Subtask:
Construct and train a hybrid CNN + LSTM model for toxicity classification.

**Reasoning**:
Import necessary Keras layers and define, compile, and train the CNN + LSTM hybrid model.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, LSTM, Dense, Dropout

# Define the CNN + LSTM model
cnn_lstm_model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=maxlen),
    Conv1D(filters=64, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=4),
    LSTM(units=64),
    Dropout(0.5),
    Dense(units=6, activation='sigmoid')
])

# Compile the model
cnn_lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
history_cnn_lstm = cnn_lstm_model.fit(
    train_padded_sequences,
    train_labels,
    epochs=1,  # Training for 1 epoch for quick demonstration
    batch_size=32,
    validation_split=0.2
)

cnn_lstm_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


3990/3990 ━━━━━━━━━━━━━━━━━━━━ 288s 71ms/step - accuracy: 0.7348 - loss: 0.0983 - val_accuracy: 0.9941 - val_loss: 0.0525


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 100, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 96, 64)         │        41,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,903,316 (30.15 MB)

 Trainable params: 2,634,438 (10.05 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 5,268,878 (20.10 MB)

## Implement Attention Mechanism

### Subtask:
Integrate an attention mechanism into one of the recurrent models (GRU or LSTM).

**Reasoning**:
Define a custom Attention layer and integrate it into the GRU model architecture.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, GRU, Dense, Dropout, Attention

# Define the Attention layer
class Attention(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(Attention, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name='attention_weight', shape=(input_shape[-1], 1),
                                 initializer='random_normal', trainable=True)
        self.b = self.add_weight(name='attention_bias', shape=(input_shape[1], 1),
                                 initializer='zeros', trainable=True)
        super(Attention, self).build(input_shape)

    def call(self, x):
        # Alignment scores
        e = tf.tanh(tf.matmul(x, self.W) + self.b)
        # Attention weights
        alpha = tf.nn.softmax(e, axis=1)
        # Context vector
        context_vector = alpha * x
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector

# Define the GRU model with Attention
inputs = Input(shape=(maxlen,))
embedding = Embedding(input_dim=max_words, output_dim=128, input_length=maxlen)(inputs)
gru_out = GRU(units=64, return_sequences=True)(embedding) # return_sequences must be True for Attention
attention_out = Attention()(gru_out)
dropout = Dropout(0.5)(attention_out)
outputs = Dense(units=6, activation='sigmoid')(dropout)

attention_gru_model = Model(inputs=inputs, outputs=outputs)

# Compile the model
attention_gru_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
history_attention_gru = attention_gru_model.fit(
    train_padded_sequences,
    train_labels,
    epochs=1,  # Training for 1 epoch for quick demonstration
    batch_size=32,
    validation_split=0.2
)

attention_gru_model.summary()

## Evaluate Models

### Subtask:
Evaluate the performance of the implemented models (GRU, CNN+LSTM, Attention GRU) using appropriate metrics on the test dataset.

**Reasoning**:
Prepare the test data for evaluation and evaluate the trained models (GRU, CNN+LSTM, Attention GRU).

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer # Explicitly import Keras Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences # Explicitly import pad_sequences
import numpy as np

# Prepare the test data for evaluation
# The test data needs to be tokenized and padded in the same way as the training data
# test_df and test_labels_df are assumed to be available from previous steps

# Ensure test_labels_df is aligned with test_df by 'id'
# We need to make sure we only evaluate on the test samples that have labels available in test_labels_df (where labels are not -1)
test_df_labeled = test_df.merge(test_labels_df, on='id')

# Filter out rows where any of the target labels are -1
for col in target_columns:
    test_df_labeled = test_df_labeled[test_df_labeled[col] != -1]

# Extract comments and labels for evaluation
test_comments = test_df_labeled['comment_text']
test_labels = test_df_labeled[target_columns].values

# Recreate the Keras tokenizer and fit it on the training data to ensure correct vocabulary
# This is a safeguard in case the original tokenizer object's state was lost or overwritten.
keras_tokenizer_eval = Tokenizer(num_words=max_words)
# Assuming train_df is available and contains the 'comment_text' column
keras_tokenizer_eval.fit_on_texts(train_df['comment_text'])


# Tokenize and pad the test comments using the Keras tokenizer
test_sequences_eval = keras_tokenizer_eval.texts_to_sequences(test_comments)
test_padded_sequences_eval = pad_sequences(test_sequences_eval, maxlen=maxlen)

print("Evaluation test sequences shape:", test_padded_sequences_eval.shape)
print("Evaluation test labels shape:", test_labels.shape)

# Evaluate the GRU model
print("\nEvaluating GRU Model:")
loss_gru, accuracy_gru = gru_model.evaluate(test_padded_sequences_eval, test_labels, verbose=0)
print(f"GRU Model Loss: {loss_gru:.4f}, Accuracy: {accuracy_gru:.4f}")

# Evaluate the CNN + LSTM model
print("\nEvaluating CNN + LSTM Model:")
loss_cnn_lstm, accuracy_cnn_lstm = cnn_lstm_model.evaluate(test_padded_sequences_eval, test_labels, verbose=0)
print(f"CNN + LSTM Model Loss: {loss_cnn_lstm:.4f}, Accuracy: {accuracy_cnn_lstm:.4f}")

# Evaluate the Attention GRU model
print("\nEvaluating Attention GRU Model:")
loss_attention_gru, accuracy_attention_gru = attention_gru_model.evaluate(test_padded_sequences_eval, test_labels, verbose=0)
print(f"Attention GRU Model Loss: {loss_attention_gru:.4f}, Accuracy: {accuracy_attention_gru:.4f}")

## Add Interpretability

### Subtask:
Implement LIME/SHAP or other techniques to add interpretability to the best-performing model (CNN+LSTM).

**Reasoning**:
Install LIME, prepare the necessary functions and data for LIME, and generate explanations for the CNN+LSTM model predictions.

In [ ]:
!pip install lime

In [ ]:
from lime.lime_text import LimeTextExplainer
from tensorflow.keras.preprocessing.text import Tokenizer # Explicitly import Keras Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences # Explicitly import pad_sequences
import numpy as np

# Assuming test_comments and cnn_lstm_model are available from previous steps
# We also need the Keras tokenizer and maxlen from preprocessing

# Recreate the Keras tokenizer object using the word_index from the original tokenizer
# This ensures we are using the correct tokenizer with the learned vocabulary
keras_tokenizer = Tokenizer(num_words=max_words)
# Assuming the original tokenizer object (which was a Keras Tokenizer)
# had its word_index populated during the fit_on_texts call
# If that object is no longer available or its state is lost, we might need to refit it.
# Assuming it's available and fitted:
# keras_tokenizer.word_index = tokenizer.word_index # This might not work if tokenizer is now HF tokenizer

# Let's refit the Keras tokenizer on the training data again to be safe
# Assuming train_df is still available
keras_tokenizer.fit_on_texts(train_df['comment_text'])


# Get the class names (toxicity types)
class_names = target_columns

# Create a LIME Text Explainer
# We need to provide the class names
explainer = LimeTextExplainer(class_names=class_names)

# Define a prediction function that takes a list of text instances
# and returns a numpy array of prediction probabilities for each class.
# The model's predict method already does this, but we need to ensure
# the input is tokenized and padded correctly within this function using the Keras tokenizer.
def predictor(texts):
    # Use the Keras tokenizer explicitly
    sequences = keras_tokenizer.texts_to_sequences(texts)
    padded_sequences = pad_sequences(sequences, maxlen=maxlen)
    return cnn_lstm_model.predict(padded_sequences)

# Generate an explanation for a specific instance (e.g., a toxic comment the model is confident about)
# Find examples of toxic comments in the labeled test set
toxic_examples = test_df_labeled[test_df_labeled['toxic'] == 1]

if not toxic_examples.empty:
    # Get predictions for these toxic examples to find one the model is confident about
    toxic_comments_text = toxic_examples['comment_text'].tolist()
    toxic_predictions = predictor(toxic_comments_text)

    # Find the index of the toxic example with the highest predicted probability for 'toxic'
    toxic_class_index = class_names.index('toxic')
    most_toxic_example_index_in_toxic_examples_df = np.argmax(toxic_predictions[:, toxic_class_index])

    # Get the original index in the test_df_labeled for the selected comment
    idx_to_explain = toxic_examples.iloc[most_toxic_example_index_in_toxic_examples_df].name
    comment_to_explain = test_df_labeled.loc[idx_to_explain, 'comment_text']

    print(f"\nGenerating explanation for comment with highest predicted 'toxic' probability: '{comment_to_explain}'")
    print(f"Predicted probabilities for this comment: {toxic_predictions[most_toxic_example_index_in_toxic_examples_df]}")


    # Generate explanation for the selected toxic comment
    # num_features is the number of words to highlight
    # num_samples is the number of perturbed samples to generate for explanation
    explanation = explainer.explain_instance(
        comment_to_explain,
        predictor,
        num_features=10, # Highlight up to 10 important words
        num_samples=5000 # Generate 5000 perturbed samples
    )

    # Print the explanation for each class
    print("\nExplanation for the prediction:")
    for class_name in class_names:
        print(f"Explanation for class '{class_name}':")
        try:
            # Get the explanation as a list of (word, weight) tuples
            explanation_list = explanation.as_list(label=class_names.index(class_name))
            print(explanation_list)
        except KeyError:
            print(f"  No explanation generated for class '{class_name}', possibly due to low predicted probability.")
        print("-" * 20)

    # Optionally, visualize the explanation (requires HTML display in supported environments)
    # from IPython.display import display, HTML
    # display(HTML(explanation.as_html()))

else:
    print("\nNo toxic comments found in the labeled test set to explain.")

191/191 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step

Generating explanation for comment with highest predicted 'toxic' probability: 'block mei could give a shit about wikipedia and your bullshit policies  
 any site that would spoil a film on opening weekend and argue about it isnt worth my time or trouble  
 its sad and pathetic that the people you have editing these pages are more concerned over getting their way than they are about ruining others information experience fuck thatand fuck this pathetic site  
 goodbye losers'
Predicted probabilities for this comment: [0.9834338  0.3466069  0.9380737  0.05626428 0.88330626 0.18615322]
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step

Explanation for the prediction:
Explanation for class 'toxic':
  No explanation generated for class 'toxic', possibly due to low predicted probability.
--------------------
Explanation for class 'severe_toxic':
[(np.str_('fuck'), 0.21596581390482553), (np.str_('pathetic'), 0.06385144050427077), (np.str_('experience'), -0.0

## Create Gradio UI

### Subtask:
Develop a Gradio interface to demonstrate the best-performing model (CNN+LSTM) and its interpretability features.

In [ ]:
import gradio as gr
import numpy as np
from lime.lime_text import LimeTextExplainer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import re
import string

# Assuming cnn_lstm_model, keras_tokenizer, maxlen, and target_columns are available

# Re-define the clean_text function used during preprocessing
def clean_text(text):
    text = text.lower()
    text = re.sub(f'[{re.escape(string.punctuation)}]', '', text)
    text = re.sub(r'\d+', '', text)
    return text

# Get the class names (toxicity types)
class_names = target_columns

# Create a LIME Text Explainer (re-using if already created, or create new)
# We need to provide the class names
explainer = LimeTextExplainer(class_names=class_names)

# Define a prediction function that takes a list of text instances
# and returns a numpy array of prediction probabilities for each class.
# The model's predict method already does this, but we need to ensure
# the input is tokenized and padded correctly within this function using the Keras tokenizer.
def predictor(texts):
    # Use the Keras tokenizer explicitly
    sequences = keras_tokenizer.texts_to_sequences(texts)
    padded_sequences = pad_sequences(sequences, maxlen=maxlen)
    return cnn_lstm_model.predict(padded_sequences)

# Define a function for Gradio that combines prediction and explanation
def analyze_toxicity(comment):
    # Get toxicity predictions
    predictions = predict_toxicity(comment)

    # Generate LIME explanation HTML
    explanation_html = explain_toxicity(comment)

    return predictions, explanation_html


# Define a prediction function for Gradio
# This function will take a text comment as input
# It needs to return probabilities for each class in a format Gradio understands (e.g., dictionary)
def predict_toxicity(comment):
    # Preprocess the comment
    cleaned_comment = clean_text(comment)
    sequences = keras_tokenizer.texts_to_sequences([cleaned_comment])
    padded_sequences = pad_sequences(sequences, maxlen=maxlen)

    # Get model predictions
    predictions = cnn_lstm_model.predict(padded_sequences)[0] # Get predictions for the single input

    # Format predictions for Gradio output
    results = {}
    for i, class_name in enumerate(class_names):
        results[class_name] = float(predictions[i]) # Convert to float for JSON compatibility

    return results

# Define a function to generate LIME explanations for Gradio
# This function will take a text comment as input and return the LIME explanation
def explain_toxicity(comment):
    # Preprocess the comment
    cleaned_comment = clean_text(comment)

    # Define the predictor function for LIME
    def lime_predictor(texts):
        sequences = keras_tokenizer.texts_to_sequences(texts)
        padded_sequences = pad_sequences(sequences, maxlen=maxlen)
        return cnn_lstm_model.predict(padded_sequences)

    # Generate LIME explanation for the cleaned comment
    explanation = explainer.explain_instance(
        cleaned_comment,
        lime_predictor,
        num_features=10, # Highlight up to 10 important words
        num_samples=1000 # Use fewer samples for faster explanation in UI
    )

    # Format explanation for Gradio HTML output
    # LIME's as_html() method provides a ready-to-use HTML string
    return explanation.as_html()


In [ ]:
# Modify the explain_toxicity function to return explanation data for heatmap
def explain_toxicity_heatmap(comment):
    cleaned_comment = clean_text(comment)
    def lime_predictor(texts):
        sequences = keras_tokenizer.texts_to_sequences(texts)
        padded_sequences = pad_sequences(sequences, maxlen=maxlen)
        return cnn_lstm_model.predict(padded_sequences)

    explanation = explainer.explain_instance(
        cleaned_comment,
        lime_predictor,
        num_features=20, # Increase num_features to highlight more words
        num_samples=1000
    )

    # Return explanation data as a list of (word, weight) tuples for each class
    explanation_data = {}
    for i, class_name in enumerate(class_names):
        try:
            explanation_data[class_name] = explanation.as_list(label=i)
        except KeyError:
            explanation_data[class_name] = [] # Handle cases where no explanation is available for a class

    return cleaned_comment, explanation_data

# Define a function to generate HTML with heatmaps
def generate_heatmap_html(comment, explanation_data):
    html_output = "<h4>LIME Explanation Heatmap</h4>"
    for class_name, exp_list in explanation_data.items():
        html_output += f"<h5>Class: {class_name}</h5>"
        if not exp_list:
            html_output += "<p>No explanation available for this class.</p>"
            continue

        # Create a dictionary of word weights for the current class
        word_weights = {word: weight for word, weight in exp_list}

        # Split the comment into words
        words = comment.split()

        # Generate HTML with colored words based on weights
        colored_text = ""
        for word in words:
            cleaned_word = re.sub(f'[{re.escape(string.punctuation)}]', '', word.lower()) # Clean word for lookup
            weight = word_weights.get(cleaned_word, 0) # Get weight, default to 0 if not in explanation

            # Normalize weight to a 0-1 scale for color intensity
            # We need a range for weights; a simple approach is to take absolute value
            # and scale. For more nuanced scaling, consider the min/max weights across all explanations.
            # For simplicity here, we'll use a basic scaling based on absolute value.
            abs_weight = abs(weight)
            # You might want to set a max weight or use a different scaling method
            # For this example, let's assume weights are roughly in a certain range
            # and scale accordingly. A simple approach:
            max_abs_weight = max(abs(w) for _, w in exp_list) if exp_list else 1.0
            intensity = min(abs_weight / max_abs_weight, 1.0) # Cap intensity at 1.0

            # Determine color based on weight (e.g., red for positive, blue for negative)
            if weight > 0:
                # Red shades for positive weights (indicating contribution to the class)
                color = f"rgba(255, 0, 0, {intensity})"
            elif weight < 0:
                # Blue shades for negative weights (indicating contribution against the class)
                color = f"rgba(0, 0, 255, {intensity})"
            else:
                color = "transparent" # No background for words with zero weight

            colored_text += f'<span style="background-color: {color};">{word}</span> '

        html_output += f"<p>{colored_text.strip()}</p>"

    return html_output

# Modify the analyze_toxicity function to return both prediction and heatmap HTML
def analyze_toxicity_with_heatmap(comment):
    predictions = predict_toxicity(comment)
    cleaned_comment, explanation_data = explain_toxicity_heatmap(comment)
    heatmap_html = generate_heatmap_html(cleaned_comment, explanation_data)
    return predictions, heatmap_html


# Recreate the Gradio interface to use the updated function and output component
iface_heatmap = gr.Interface(
    fn=analyze_toxicity_with_heatmap, # Updated function
    inputs=gr.Textbox(lines=5, label="Enter a comment"), # Input component (text box)
    outputs=[
        gr.Label(num_top_classes=None, label="Toxicity Prediction"), # Output for predictions
        gr.HTML(label="LIME Explanation Heatmap") # Output for LIME heatmap HTML
    ],
    title="Toxicity Classification and Interpretability with Heatmap",
    description="Enter a comment to get toxicity predictions and a LIME explanation heatmap."
)

# Launch the updated Gradio interface
iface_heatmap.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c2a84ee35b77490c1f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
